In [1]:
# 3_top_k_retrieval_ablation.py
!pip install -q torch transformers sentence-transformers faiss-cpu pandas tqdm scikit-learn rouge-score nltk bert-score sacrebleu

# ------------------- IMPORTS -------------------
import pandas as pd, numpy as np, torch, faiss, time, nltk, warnings, logging
from sentence_transformers import SentenceTransformer, util
from transformers import pipeline
from tqdm.auto import tqdm
from rouge_score import rouge_scorer
from nltk.translate.meteor_score import meteor_score
from nltk.tokenize import word_tokenize
from bert_score import score as bert_score
from sacrebleu.metrics import BLEU

# Reduce HuggingFace model init warnings
logging.getLogger("transformers.modeling_utils").setLevel(logging.ERROR)

# NLTK downloads
nltk.download('punkt', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

warnings.filterwarnings('ignore')

# ------------------- DATA -------------------
df = pd.read_csv('/kaggle/input/mlops-amazon/amazon.csv')

documents = [
    f"""Product: {r['product_name']}
Price: {r['discounted_price']} | Rating: {r['rating']} ({r['rating_count']} reviews)
Description: {r['about_product']}""" 
    for _, r in df.iterrows()
]

TEST_QUERIES = [
{
"query": "Recommend a good fast charging USB-C cable under 300 rupees",
"reference": "The boAt A400 at ₹299 is recommended, supporting 24W-25W fast charging and having a slightly higher customer rating. For a tighter budget, the pTron Solero TB301 at ₹149 supports 15W fast charging."
},
{
"query": "Which cable has the highest rating and supports 60W charging?",
"reference": "The Belkin USB-C to USB-C Fast Charging Cable (60W PD) is the best match, offering a strong 4.5-star rating along with full 60W fast-charging support."
},
{
"query": "What is the best iPhone lightning cable in the list?",
"reference": "For super-fast charging with a USB-C adapter, the Belkin Lightning to USB-C is best. For a reliable and durable standard cable with a good warranty, the Duracell USB-A to Lightning is a great choice. The Hi-Mobiler is the most budget-friendly option."
},
{
"query": "Suggest me some good long lasting headphones",
"reference": "The boAt Bassheads 100 in-ear wired earphones are recommended for longevity, featuring a premium coated wire for sturdiness and a 1-year warranty. They have a 4.1-star rating from over 360,000 reviews and cost between ₹349-₹379."
}
]

# ------------------- METRICS CLASS -------------------
class Metrics:
    def __init__(self):
        self.rouge = rouge_scorer.RougeScorer(['rouge1','rougeL'], use_stemmer=True)
        self.bleu = BLEU(effective_order=True)
        self.embedder = SentenceTransformer('all-MiniLM-L6-v2')

    def all(self, pred, ref, ctx):
        r = self.rouge.score(ref, pred)
        metrics = {
            'rouge_1_f1': r['rouge1'].fmeasure,
            'rouge_l_f1': r['rougeL'].fmeasure,
            'bleu': self.bleu.sentence_score(pred, [ref]).score / 100,
            'meteor': meteor_score([word_tokenize(ref.lower())], word_tokenize(pred.lower())),
        }
        P, R, F = bert_score([pred], [ref], model_type="microsoft/deberta-large-mnli", verbose=False)
        metrics['bert_f1'] = F.mean().item()

        e1 = self.embedder.encode(pred)
        e2 = self.embedder.encode(ref)
        metrics['emb_sim'] = util.cos_sim(e1, e2).item()

        c_emb = self.embedder.encode(ctx)
        metrics['faith'] = min(1.0, 0.7 * util.cos_sim(e1, c_emb).item() + 0.3)

        return metrics

    def composite(self, m):
        w = {
            'rouge_1_f1': 0.1,
            'rouge_l_f1': 0.1,
            'bleu': 0.1,
            'meteor': 0.15,
            'bert_f1': 0.25,
            'emb_sim': 0.2,
            'faith': 0.1
        }
        return sum(m[k] * w[k] for k in w)

metrics_calc = Metrics()

# ------------------- RAG CLASS -------------------
class RAG:
    def __init__(self, emb_name, generator):
        self.emb_name = emb_name
        self.generator = generator

        print(f"Loading embedding model: {emb_name}")
        self.embedder = SentenceTransformer(emb_name)
        dim = self.embedder.encode(["test"]).shape[1]
        self.index = faiss.IndexFlatIP(dim)

        print(f"Embedding {len(documents)} documents...")
        batches = [documents[i:i+32] for i in range(0, len(documents), 32)]
        for b in tqdm(batches, desc="Indexing"):
            embs = self.embedder.encode(b, normalize_embeddings=True)
            self.index.add(embs)

    def retrieve(self, q, k):
        qe = self.embedder.encode([q], normalize_embeddings=True)
        D, I = self.index.search(qe, k)
        ctx = "\n\n".join([documents[i] for i in I[0]])
        return ctx

    def generate(self, q, ctx):
        prompt = f"Context:\n{ctx}\n\nQuestion: {q}\nAnswer:"
        out = self.generator(
            prompt,
            max_new_tokens=512,
            temperature=0.7,
            top_p=0.95,
            top_k=50,
            do_sample=True
        )[0]['generated_text']
        ans = out.split("Answer:")[-1].strip()
        return ans

# ------------------- LOAD GENERATOR -------------------
GEN_MODEL = "Qwen/Qwen2.5-7B-Instruct"
print("\nLoading generator...")
generator = pipeline(
    "text-generation",
    model=GEN_MODEL,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

# ------------------- LOAD RAG -------------------
EMBEDDING_MODEL = "BAAI/bge-small-en-v1.5"
print("\nLoading RAG with fixed models...")
rag = RAG(EMBEDDING_MODEL, generator)

# ------------------- TOP-K RETRIEVAL EXPERIMENT -------------------
results = []
TOP_K_VALUES = [1, 3, 5, 8, 10, 15]

for k in TOP_K_VALUES:
    print(f"\n{'='*80}\nTESTING TOP-K: {k}\n{'='*80}")
    for qd in TEST_QUERIES:
        ctx = rag.retrieve(qd["query"], k=k)
        ans = rag.generate(qd["query"], ctx)
        m = metrics_calc.all(ans, qd["reference"], ctx)
        m['composite'] = metrics_calc.composite(m)
        results.append({**m, "top_k": k, "query": qd["query"][:60]})
        print("\n------------------------------------------------------------")
        print(f"Top-K: {k}")
        print(f"Query: {qd['query']}")
        print("\nGenerated Answer:")
        print(ans)
        print(f"\nComposite Score: {m['composite']:.4f}")
        print("------------------------------------------------------------\n")

df_out = pd.DataFrame(results)
summary = df_out.groupby("top_k")["composite"].mean().sort_values(ascending=False)

print("\n================ FINAL SUMMARY ================\n")
print("Average Composite Scores by Top-K:")
print(summary)

best_k = summary.idxmax()
best_score = summary.max()
print(f"\n🏆 Best Top-K: {best_k} → Composite Score: {best_score:.4f}")

df_out.to_csv("3_top_k_retrieval_ablation.csv", index=False)
print("\nTop-K ablation results saved → 3_top_k_retrieval_ablation.csv")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 109.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 77.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 34.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 33.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 96.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 94.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

2025-12-05 07:13:08.194035: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764918788.585016      20 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764918788.710067      20 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

...


------------------------------------------------------------
Top-K: 15
Query: Which cable has the highest rating and supports 60W charging?

Generated Answer:
The Ambrane 60W / 3A Type C Fast Charging Unbreakable 1.5m L Shaped Braided Cable (ABLC10, Black) has the highest rating of 4.0 and supports 60W charging. However, it's worth noting that the MI Xiaomi USB Type C HYperCharge Cable 6A 100cm also supports 120W hypercharging, which is higher than 60W.

To summarize, the Ambrane cable is the one with the highest rating among those that support 60W charging. For 120W hypercharging, the MI Xiaomi cable would be the better choice. However, the Ambrane cable might be preferred if 60W is sufficient and you value a higher rating. 

Both have similar features like durability, fast charging, and data transfer capabilities, so the decision may come down to your specific needs and budget. 

Would you like more information on any particular cable or need help with something else? 😊👍🏼👍🏼👍🏼👍🏼👍🏼👍🏼👍

The following layers were not sharded: encoder.layer.*.output.dense.weight, encoder.layer.*.attention.self.pos_q_proj.bias, encoder.rel_embeddings.weight, encoder.layer.*.attention.output.dense.bias, encoder.layer.*.attention.self.pos_q_proj.weight, encoder.layer.*.attention.self.v_bias, encoder.layer.*.attention.self.in_proj.weight, encoder.layer.*.intermediate.dense.bias, encoder.layer.*.attention.output.dense.weight, encoder.layer.*.attention.self.q_bias, embeddings.LayerNorm.weight, encoder.layer.*.attention.output.LayerNorm.bias, encoder.layer.*.output.LayerNorm.bias, embeddings.word_embeddings.weight, encoder.layer.*.attention.output.LayerNorm.weight, encoder.layer.*.attention.self.pos_proj.weight, encoder.layer.*.output.dense.bias, encoder.layer.*.output.LayerNorm.weight, encoder.layer.*.intermediate.dense.weight, embeddings.LayerNorm.bias



------------------------------------------------------------
Top-K: 15
Query: What is the best iPhone lightning cable in the list?

Generated Answer:
Determining the "best" iPhone lightning cable depends on various factors such as price, features, compatibility, and user reviews. Here's a summary of each product to help you decide:

1. **Hi-Mobiler iPhone Charger Lightning Cable**
   - **Pros:** Internationally certified materials, over 15,000 bend cycles, 2-pack for versatility.
   - **Cons:** Relatively low rating compared to others.

2. **Belkin Apple Certified Lightning To Type C Cable**
   - **Pros:** Supports USB Power Delivery for fast charging, 10,000+ bend cycles, longer than typical cables.
   - **Cons:** More expensive, primarily for newer iPhones.

3. **Duracell USB Lightning Apple Certified (MFi) Braided Sync & Charge Cable**
   - **Pros:** Tangle-free, durable, fast and stable data transmission, 2-year warranty.
   - **Cons:** Lower bend cycles compared to some other opt

The following layers were not sharded: encoder.layer.*.output.dense.weight, encoder.layer.*.attention.self.pos_q_proj.bias, encoder.rel_embeddings.weight, encoder.layer.*.attention.output.dense.bias, encoder.layer.*.attention.self.pos_q_proj.weight, encoder.layer.*.attention.self.v_bias, encoder.layer.*.attention.self.in_proj.weight, encoder.layer.*.intermediate.dense.bias, encoder.layer.*.attention.output.dense.weight, encoder.layer.*.attention.self.q_bias, embeddings.LayerNorm.weight, encoder.layer.*.attention.output.LayerNorm.bias, encoder.layer.*.output.LayerNorm.bias, embeddings.word_embeddings.weight, encoder.layer.*.attention.output.LayerNorm.weight, encoder.layer.*.attention.self.pos_proj.weight, encoder.layer.*.output.dense.bias, encoder.layer.*.output.LayerNorm.weight, encoder.layer.*.intermediate.dense.weight, embeddings.LayerNorm.bias



------------------------------------------------------------
Top-K: 15
Query: Suggest me some good long lasting headphones

Generated Answer:
Based on the descriptions provided, here are some good long-lasting options for headphones:

1. **boAt Rockerz 450 Bluetooth On Ear Headphones with Mic**
   - **Price:** ₹1,220
   - **Battery Life:** 15 hours
   - **Drivers:** 40mm dynamic drivers
   - **Features:** Ergonomic design, comfortable padded earcushions, dual connectivity modes, and 1-year warranty.
   
2. **Noise Buds VS402 Truly Wireless in Ear Earbuds**
   - **Price:** ₹1,799
   - **Battery Life:** 35 hours (total)
   - **Drivers:** 10mm driver
   - **Features:** Environmental noise cancellation, quick charge, hyper sync, and 1-year warranty.
   
3. **boAt Rockerz 400 Bluetooth On Ear Headphones With Mic**
   - **Price:** ₹1,399
   - **Battery Life:** 8 hours
   - **Drivers:** 40mm drivers
   - **Features:** Lightweight, ergonomic design, easy access integrated controls, dual conne